# Project 1 — Celebrity Detection
## Milestone 3: YOLOv8 Fine-Tuning

**IE 7615** — Project 1, Milestone 3

**David Fung**

This milestone fine-tunes a YOLOv8 model on the synthetic multi-face dataset from Milestone 2. Starting from the `yolov8n.pt` pretrained weights, the model is trained for 50 epochs on 640×640 images containing 2–5 faces per image from the 5 selected celebrity identities. Training progress is visualized and final test set performance is reported.

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ultralytics
from ultralytics import YOLO
import torch

import tensorflow as tf

print(f"Ultralytics: {ultralytics.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"TensorFlow: {tf.__version__}")

Ultralytics: 8.4.163
PyTorch: 2.14.0+cu132
TensorFlow: 2.13.1


In [3]:
# PROJECT CONSTANTS
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
tf.keras.utils.set_random_seed(GLOBAL_SEED)

def set_all_seeds():
    np.random.seed(GLOBAL_SEED)
    torch.manual_seed(GLOBAL_SEED)
    tf.random.set_seed(GLOBAL_SEED)

# YOLOv8 parameters
IMG_SIZE = 640
BATCH = 16
# Set to 10 or 20 for testing and final 50
EPOCHS = 10
DATA_YAML = "data/synthetic_multi_face/data.yaml"
PRETRAINED_WEIGHTS = "yolov8n.pt"
NUM_CLASSES = 5
CLASS_NAMES = ['7', '797', '2619', '4428', '7007']
PROJECT_NAME = "celebrity_detection"
RUN_NAME = "yolov8n_finetune"


## Helper Functions

Reproducible seeding and training metrics visualization.

In [4]:
def plot_history(results, title="YOLOv8 Training Results"):
    """Plot training/validation metrics from YOLO results object."""
    # Get the results dict from the Results object
    if hasattr(results, 'results_dict'):
        r = results.results_dict
    elif isinstance(results, dict):
        r = results
    else:
        r = results[0] if isinstance(results, list) else results
    
    # Get number of epochs
    num_epochs = len(r.get('train/box_loss', []))
    if num_epochs == 0:
        print("No training history found in results")
        return
    
    epochs = list(range(1, num_epochs + 1))
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(title, fontsize=16)
    
    # Plot 1: Box Loss
    axes[0, 0].plot(epochs, r.get('train/box_loss', []), 'b-', label='Train')
    axes[0, 0].plot(epochs, r.get('val/box_loss', []), 'r-', label='Val')
    axes[0, 0].set_title('Box Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Cls Loss
    axes[0, 1].plot(epochs, r.get('train/cls_loss', []), 'b-', label='Train')
    axes[0, 1].plot(epochs, r.get('val/cls_loss', []), 'r-', label='Val')
    axes[0, 1].set_title('Classification Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: DFL Loss
    axes[0, 2].plot(epochs, r.get('train/dfl_loss', []), 'b-', label='Train')
    axes[0, 2].plot(epochs, r.get('val/dfl_loss', []), 'r-', label='Val')
    axes[0, 2].set_title('DFL Loss')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Loss')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Plot 4: Metrics - Precision, Recall, mAP
    axes[1, 0].plot(epochs, r.get('metrics/precision(B)', []), 'g-', label='Precision')
    axes[1, 0].plot(epochs, r.get('metrics/recall(B)', []), 'b-', label='Recall')
    axes[1, 0].plot(epochs, r.get('metrics/mAP_0.5(B)', []), 'r-', label='mAP@0.5')
    axes[1, 0].set_title('Metrics')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 5: mAP at different IoU thresholds
    axes[1, 1].plot(epochs, r.get('metrics/mAP_0.5(B)', []), 'r-', label='mAP@0.5')
    axes[1, 1].plot(epochs, r.get('metrics/mAP_0.5:0.95(B)', []), 'purple', label='mAP@0.5:0.95')
    axes[1, 1].set_title('mAP at IoU Thresholds')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('mAP')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Plot 6: Learning rate (if available)
    lr_keys = [k for k in r.keys() if 'lr' in k.lower()]
    if lr_keys:
        axes[1, 2].plot(epochs, r.get(lr_keys[0], []), 'orange')
        axes[1, 2].set_title('Learning Rate')
        axes[1, 2].set_xlabel('Epoch')
        axes[1, 2].set_ylabel('LR')
    else:
        axes[1, 2].text(0.5, 0.5, 'LR data not available', ha='center', va='center', transform=axes[1, 2].transAxes)
        axes[1, 2].set_title('Learning Rate')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Task 1 — Verify Dataset

Check that the synthetic dataset from Milestone 2 is available and show its contents.

In [9]:
import os

# Check that the dataset exists
assert os.path.exists(DATA_YAML), f"Dataset not found: {DATA_YAML}"

with open(DATA_YAML) as f:
    print(f.read())


path: data/synthetic_multi_face
train: images/train
val: images/val
test: images/test

nc: 5
names:
  0: 7
  1: 797
  2: 2619
  3: 4428
  4: 7007



## Task 2 — Load Model

Initialize the YOLOv8 model with pretrained weights.

In [10]:
model = YOLO(PRETRAINED_WEIGHTS)

# Show model architecture
print(model)


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0

## Task 3 — Train Model

Fine-tune the YOLOv8 model on the synthetic multi-face dataset.

In [ ]:
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMG_SIZE,
    project=PROJECT_NAME,
    name=RUN_NAME,
    exist_ok=True
)

print(f"\nTraining complete!")
print(f"Model saved to: runs/detect/{PROJECT_NAME}/{RUN_NAME}/weights/best.pt")


Ultralytics 8.4.163  Python-3.10.21 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16275MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/synthetic_multi_face/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_finetune, nbs=64,

## Task 4 — Plot Training Results

Visualize training and validation metrics over epochs.

In [ ]:
plot_history(results, title="YOLOv8 Fine-Tuning: Celebrity Detection")


## Task 5 — Evaluate Test Set

Run validation on the test split to get final performance metrics.

In [ ]:
test_results = model.val(data=DATA_YAML, split='test', imgsz=IMG_SIZE)

# Print key metrics
print(f"\n{'='*50}")
print(f"Test Set Metrics")
print(f"{'='*50}")
print(f"mAP@0.5:       {test_results.box.map:.4f}")
print(f"mAP@0.5:0.95: {test_results.box.map5095:.4f}")
print(f"Precision:     {test_results.box.mp:.4f}")
print(f"Recall:        {test_results.box.mr:.4f}")
print(f"{'='*50}")


## Task 6 — Save & Summarize

Locate the best model weights and show a final summary table.

In [ ]:
# Locate best weights
best_weights = os.path.join("runs", "detect", PROJECT_NAME, RUN_NAME, "weights", "best.pt")

print(f"Best model saved to: {best_weights}")
print(f"File size: {os.path.getsize(best_weights) / 1e6:.2f} MB")

# Create summary table
summary = pd.DataFrame({
    'Metric': ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall'],
    'Value': [test_results.box.map, test_results.box.map5095, test_results.box.mp, test_results.box.mr]
})

print(f"\nFinal Test Set Performance Summary:")
print(summary.to_string(index=False))


## Summary

**Model:** YOLOv8 Nano (fine-tuned)

**Dataset:** 200 synthetic multi-face images (Milestone 2)

**Training:** 50 epochs, batch size 16, 640×640 images

**Final Metrics:**
- mAP@0.5: See test results above
- mAP@0.5:0.95: See test results above
- Precision & Recall: See test results above

**Next Step:** Open `Milestone4_Pipeline_Integration.ipynb` to integrate the trained model into the `CelebrityPipeline` and run a live demo on new images.